In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score,accuracy_score
from sklearn.pipeline import Pipeline

In [2]:
data = pd.read_csv('ipl_matches.csv')

In [3]:
data.head()

,team1,team2,city,venue,season,toss_winner,toss_decision,winner,win_by_runs,win_by_wickets
0,Chennai Super Kings,Kings XI Punjab,Chennai,MA Chidambaram Stadium,2019,Chennai Super Kings,bat,Chennai Super Kings,22,0
1,Royal Challengers Bangalore,Kolkata Knight Riders,Bengaluru,M.Chinnaswamy Stadium,2019,Kolkata Knight Riders,field,Kolkata Knight Riders,0,5
2,Royal Challengers Bangalore,Kings XI Punjab,Bangalore,M Chinnaswamy Stadium,2009/10,Kings XI Punjab,bat,Royal Challengers Bangalore,0,8
3,Royal Challengers Bangalore,Pune Warriors,Bangalore,M Chinnaswamy Stadium,2012,Pune Warriors,bat,Royal Challengers Bangalore,0,6
4,Delhi Daredevils,Royal Challengers Bangalore,Delhi,Feroz Shah Kotla,2007/08,Royal Challengers Bangalore,field,Delhi Daredevils,10,0


In [4]:
team_name_mapping = {'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings',
                     'Royal Challengers Bangalore': 'Royal Challengers Bengaluru','Rising Pune Supergiants': 'Rising Pune Supergiant',
                     'Gujarat Lions':'Gujarat Titans'}

In [5]:
data['team1'] = data['team1'].replace(team_name_mapping)

In [6]:
defunct_teams = ['Deccan Chargers', 'Kochi Tuskers Kerala','Rising Pune Supergiant','Pune Warriors']

In [7]:
data = data[~data['team1'].isin(defunct_teams)]

In [8]:
data['team2'] = data['team2'].replace(team_name_mapping)

In [9]:
data = data[~data['team2'].isin(defunct_teams)]

In [10]:
data['toss_winner'] = data['toss_winner'].replace(team_name_mapping)
data['winner'] = data['winner'].replace(team_name_mapping)

In [11]:
data = data.drop(columns=['city'])

In [12]:
venue_mapping = {'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium','M Chinnaswamy Stadium, Bengaluru': 'M Chinnaswamy Stadium',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium','MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium','Arun Jaitley Stadium, Delhi': 'Arun Jaitley Stadium',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium','Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'Rajiv Gandhi International Stadium',
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium','Eden Gardens, Kolkata': 'Eden Gardens',
    'Sawai Mansingh Stadium, Jaipur': 'Sawai Mansingh Stadium','Dr DY Patil Sports Academy, Mumbai': 'Dr DY Patil Sports Academy',
    'Brabourne Stadium, Mumbai': 'Brabourne Stadium','Punjab Cricket Association Stadium, Mohali': 'IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'IS Bindra Stadium','Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium': 'IS Bindra Stadium','Sardar Patel Stadium, Motera': 'Narendra Modi Stadium',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'Ekana Cricket Stadium',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam':'ACA-VDCA Cricket Stadium',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala':'HPCA Stadium',
    'Maharashtra Cricket Association Stadium, Pune':'MCA Stadium','Maharashtra Cricket Association Stadium':'MCA Stadium',
    "Sheikh Zayed Stadium": "Zayed Cricket Stadium, Abu Dhabi","Narendra Modi Stadium": "Narendra Modi Stadium, Ahmedabad",
    "HPCA Stadium": "Himachal Pradesh Cricket Association Stadium","Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh":
    "Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur","ACA-VDCA Cricket Stadium":"Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium"}

In [13]:
data['venue'] = data['venue'].replace(venue_mapping)

In [14]:
top_venues = data['venue'].value_counts().nlargest(20).index

In [15]:
data['venue'] = data['venue'].apply(lambda x: x if x in top_venues else 'Other')

In [16]:
data = data.drop(columns=['season'])

In [17]:
data.head()

,team1,team2,venue,toss_winner,toss_decision,winner,win_by_runs,win_by_wickets
0,Chennai Super Kings,Punjab Kings,MA Chidambaram Stadium,Chennai Super Kings,bat,Chennai Super Kings,22,0
1,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Kolkata Knight Riders,field,Kolkata Knight Riders,0,5
2,Royal Challengers Bengaluru,Punjab Kings,M Chinnaswamy Stadium,Punjab Kings,bat,Royal Challengers Bengaluru,0,8
4,Delhi Capitals,Royal Challengers Bengaluru,Arun Jaitley Stadium,Royal Challengers Bengaluru,field,Delhi Capitals,10,0
5,Delhi Capitals,Royal Challengers Bengaluru,Arun Jaitley Stadium,Delhi Capitals,field,Royal Challengers Bengaluru,4,0


Feature Engineering

In [18]:
data['toss_win_team1'] = (data['toss_winner'] == data['team1']).astype(int)

In [19]:
data['toss_decision'] = data['toss_decision'].map({'bat': 0,'field': 1})

In [20]:
data['team1_batting_first'] = (((data['toss_winner'] == data['team1']) & (data['toss_decision'] == 0)) |((data['toss_winner'] == data['team2']) & (data['toss_decision'] == 1))).astype(int)

In [21]:
team_win_rate = data.groupby('winner').size() / data.groupby('team1').size()
data['strength_diff'] = data['team1'].map(team_win_rate) - data['team2'].map(team_win_rate)

In [22]:
data['match_context'] = data['team1'].astype(str) + '_' + data['team2'].astype(str) + '_' + data['venue'].astype(str) + '_' + data['toss_winner'].astype(str)

In [23]:
context_wins = data.groupby(['match_context', 'winner']).size().unstack(fill_value=0)
context_matches = data.groupby('match_context').size()

In [24]:
def get_context_win_rate(row, team):
    ctx = row['match_context']
    t = row[team]
    if ctx in context_wins.index and t in context_wins.columns:
        return context_wins.loc[ctx, t] / context_matches[ctx]
    return 0.5

In [25]:
data['team1_context_win_prob'] = data.apply(lambda r: get_context_win_rate(r, 'team1'), axis=1)
data['team2_context_win_prob'] = data.apply(lambda r: get_context_win_rate(r, 'team2'), axis=1)

In [26]:
data = data.drop(columns=['match_context'])

In [27]:
encode = {'team1': {'Chennai Super Kings':1,'Royal Challengers Bengaluru':2,'Delhi Capitals':3,'Sunrisers Hyderabad':4,'Kolkata Knight Riders':5,'Lucknow Super Giants':6,'Mumbai Indians':7,'Punjab Kings':8,'Rajasthan Royals':9,'Gujarat Titans':10},
          'team2': {'Chennai Super Kings':1,'Royal Challengers Bengaluru':2,'Delhi Capitals':3,'Sunrisers Hyderabad':4,'Kolkata Knight Riders':5,'Lucknow Super Giants':6,'Mumbai Indians':7,'Punjab Kings':8,'Rajasthan Royals':9,'Gujarat Titans':10},
          'toss_winner': {'Chennai Super Kings':1,'Royal Challengers Bengaluru':2,'Delhi Capitals':3,'Sunrisers Hyderabad':4,'Kolkata Knight Riders':5,'Lucknow Super Giants':6,'Mumbai Indians':7,'Punjab Kings':8,'Rajasthan Royals':9,'Gujarat Titans':10},
          'winner': {'Chennai Super Kings':1,'Royal Challengers Bengaluru':2,'Delhi Capitals':3,'Sunrisers Hyderabad':4,'Kolkata Knight Riders':5,'Lucknow Super Giants':6,'Mumbai Indians':7,'Punjab Kings':8,'Rajasthan Royals':9,'Gujarat Titans':10}}

In [28]:
data.replace(encode, inplace=True)

/tmp/ipykernel_6090/3896906714.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace(encode, inplace=True)


In [29]:
cat_cols = ['venue']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

In [30]:
data = data.drop(columns=['win_by_runs','win_by_wickets'])

In [31]:
data = data.dropna()

In [32]:
data.head()

,team1,team2,venue,toss_winner,toss_decision,winner,toss_win_team1,team1_batting_first,strength_diff,team1_context_win_prob,team2_context_win_prob
0,1,8,10,1,0,1,1,1,0.211982,0.750000,0.250000
1,2,5,9,5,1,5,0,1,-0.171577,0.000000,1.000000
2,2,8,9,8,0,2,0,0,0.018808,0.555556,0.444444
4,3,2,0,2,1,3,0,1,0.016839,0.166667,0.833333
5,3,2,0,3,1,2,1,0,0.016839,0.500000,0.500000


In [33]:
X = data.drop(columns=['winner'])
y = data['winner']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [43]:
algorithms = {
    'Logistic Regression': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(solver='saga', max_iter=10000))]),
        "params": {
            "classifier__penalty": ['l1', 'l2', 'elasticnet'],
            "classifier__C": [0.01, 0.1, 1, 10, 100],
            "classifier__l1_ratio": [0.0, 0.5, 1.0]
        }
    },
    'Decision Tree': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', tree.DecisionTreeClassifier())]),
        "params": {
            "classifier__criterion": ['gini', 'entropy'],
            "classifier__max_depth": [None, 5, 10, 15, 20, 30],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 4, 8]
        }
    },
    'Random Forest': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', RandomForestClassifier(random_state=42))]),
        "params": {
            "classifier__n_estimators": [100, 200, 300, 400],
            "classifier__max_features": ["sqrt", "log2", None],
            "classifier__max_depth": [None, 10, 20, 30, 40],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__bootstrap": [True, False]
        }
    },
    'NaiveBayes': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', GaussianNB())]),
        "params": {
            "classifier__var_smoothing": np.logspace(0, -9, num=10)
        }
    },
    'K-Nearest Neighbors': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', KNeighborsClassifier())]),
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 10, 15, 20],
            "classifier__weights": ["uniform", "distance"],
            "classifier__p": [1, 2]
        }
    },
    'Gradient Boost': {
        "model": Pipeline([('scaler', StandardScaler()), ('classifier', GradientBoostingClassifier(random_state=42))]),
        "params": {
            "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
            "classifier__n_estimators": [100, 200, 300],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__max_depth": [3, 5, 10],
            "classifier__subsample": [0.8, 0.9, 1.0]
        }
    }
}

In [49]:
prediction_models = {}
model_details = []

for model_name, values in algorithms.items():
    best_score = float('-inf')
    best_rscv = None

    try:
        rscv = RandomizedSearchCV(estimator=values["model"],param_distributions=values["params"],cv=5,n_iter=30,n_jobs=-1,verbose=0,random_state=42)
        rscv.fit(X_train, y_train)

        if rscv.best_score_ > best_score:
            best_score = rscv.best_score_
            best_rscv = rscv

    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"Error with {model_name} (Prediction): {e}")
        break

    if best_rscv:
        prediction_models[model_name] = best_rscv
        model_details.append({"Model Name": model_name,"Best Score": best_score,"Best Parameters": best_rscv.best_params_})
        print(f"{model_name}: Best Score = {best_score:.4f}")
    else:
        print(f"{model_name}: No valid configuration found.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


Logistic Regression: Best Score = 0.2456
Decision Tree: Best Score = 0.5329
Random Forest: Best Score = 0.7127


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 10 is smaller than n_iter=30. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


NaiveBayes: Best Score = 0.2810


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 24 is smaller than n_iter=30. Running 24 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


K-Nearest Neighbors: Best Score = 0.5506
Gradient Boost: Best Score = 0.7481


In [50]:
pd.set_option('display.max_colwidth', None)
pd.DataFrame(model_details)

,Model Name,Best Score,Best Parameters
0,Logistic Regression,0.245570,"{'classifier__penalty': 'l2', 'classifier__l1_ratio': 1.0, 'classifier__C': 1}"
1,Decision Tree,0.532911,"{'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 15, 'classifier__criterion': 'entropy'}"
2,Random Forest,0.712658,"{'classifier__n_estimators': 100, 'classifier__min_samples_split': 5, 'classifier__max_features': 'log2', 'classifier__max_depth': 30, 'classifier__bootstrap': True}"
3,NaiveBayes,0.281013,{'classifier__var_smoothing': 1.0}
4,K-Nearest Neighbors,0.550633,"{'classifier__weights': 'distance', 'classifier__p': 1, 'classifier__n_neighbors': 20}"
5,Gradient Boost,0.748101,"{'classifier__subsample': 0.9, 'classifier__n_estimators': 300, 'classifier__min_samples_split': 10, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01}"


In [46]:
test_results = []

for model_name, model in prediction_models.items():
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    test_results.append({"Model Name": model_name,"Test Score": model.score(X_test, y_test),"Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],"F1-score": report["weighted avg"]["f1-score"]})

results_df = pd.DataFrame(test_results)

In [47]:
results_df

,Model Name,Test Score,Precision,Recall,F1-score
0,Logistic Regression,0.227273,0.197598,0.227273,0.208206
1,Decision Tree,0.595960,0.599686,0.595960,0.595218
2,Random Forest,0.707071,0.714142,0.707071,0.708501
3,NaiveBayes,0.252525,0.213204,0.252525,0.213038
4,K-Nearest Neighbors,0.525253,0.532901,0.525253,0.521314
5,Gradient Boost,0.722222,0.730976,0.722222,0.723169


In [40]:
best_model_row = results_df.loc[results_df['Test Score'].idxmax()]
best_model_name = best_model_row['Model Name']
best_model = prediction_models[best_model_name]

print(f"\nBest Model Selected: {best_model_name}")
print(f"Test Score: {best_model_row['Test Score']:.4f}")


Best Model Selected: Gradient Boost
Test Score: 0.7222
